In [1]:
# load libraies

%run py_libraries.py

/Users/4476224/.local/lib/python3.8/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.6.0 and strictly below 2.9.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.13.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/addons
  warnings.warn(


In [2]:
# loading utility files

from utility.sv_fig import savefig
# from utility.mi_score import MI_score



In [3]:
# def savefig(filename, crop = True):
#     plt.savefig('{}.pdf'.format(filename))

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)

# load data
data = pd.read_csv('data/baselinedata_37F.csv')
# data = data.loc[0:317]

print(data.shape)

(184, 38)


In [5]:
data = data.apply(pd.to_numeric) # convert all columns of Ndata to numerics

In [6]:
# count number of NCa

# NCa
data[data.CACHEXSTAGE0VIG == 0].shape

(28, 38)

In [7]:
# count number of PCa

# PCa
data[data.CACHEXSTAGE0VIG == 1].shape

(53, 38)

In [8]:
# count number of Ca

# Ca
data[data.CACHEXSTAGE0VIG == 2].shape

(103, 38)

In [9]:
# NCa v PCa
ndata_NCa_PCa = data[data["CACHEXSTAGE0VIG"] != 2]
ndata_NCa_PCa.reset_index(drop=True, inplace=True)
print(ndata_NCa_PCa.shape)

(81, 38)


In [10]:
## NCa v PCa

# # ndata_NCa_Ca = ndata_NCa_Ca.apply(pd.to_numeric) # convert all columns of Ndata to numerics

# # Replace all occurrences of 2 with 1
# ndata_NCa_Ca["CACHEXSTAGE0VIG"] = ndata_NCa_Ca["CACHEXSTAGE0VIG"].replace(2, 1)

# # # Replace all occurrences of 0 with -1
# # ndata_NCa_Ca["CACHEXSTAGE0VIG"] = ndata_NCa_Ca["CACHEXSTAGE0VIG"].replace(0, -1)




In [11]:
# count number of NCa

# NCa
ndata_NCa_PCa[ndata_NCa_PCa.CACHEXSTAGE0VIG == 0].shape

(28, 38)

In [12]:
# count number of PCa

# PCa
ndata_NCa_PCa[ndata_NCa_PCa.CACHEXSTAGE0VIG == 1].shape

(53, 38)

In [13]:
# calculate proportion of missingness in ndata_noR

# total num of NaN in the ndata_noR
total_nan_count = ndata_NCa_PCa.isna().sum().sum()

# total num of cells in ndata_noR
total_cells = ndata_NCa_PCa.size

# proportion of NaN
NaN_proportion = total_nan_count / total_cells

print('NaN_proportions:',NaN_proportion)


NaN_proportions: 0.016894087069525665


In [14]:
# table 

col_tab = ['ENA.78', 'IFN.y', 'IL.10', 'IL.6', 'IL.8', 'MCP.1', 'MDC', 'MIP.1a', 'TNF.a', 'C.peptide', 'G.CSF', 'IL.22', 'Insulin', 
           'Leptin', 'MIP.3a', 'GRO.a', 'HGF', 'MMP.2', 'Adiponectin', 'CRP', 'GDF.15', 'TIMP.1', 'TGF.B2', 'TGF.B1', 'PPAR.y', 
           'HIF.1a', 'Laminin', 'HbA1c', 'CA19.9', 'Glucose', 'HDL', 'CCK', 'LDL', 'Triglyceride', 'Albumin', 'Lumican', 'ZAG', 'CACHEXSTAGE0VIG']

categorical = ['CACHEXSTAGE0VIG']

groupby = ['CACHEXSTAGE0VIG']

myTable_demo_lg = TableOne(ndata_NCa_PCa,columns=col_tab,categorical=categorical,groupby=groupby,pval=True)

print(myTable_demo_lg.tabulate(tablefmt="latex"))





\begin{tabular}{lllllll}
\hline
                         &     & Missing   & Overall    & 0.0        & 1.0        & P-Value   \\
\hline
 n                       &     &           & 81         & 28         & 53         &           \\
 ENA.78, mean (SD)       &     & 0         & 13.2 (1.2) & 13.2 (1.1) & 13.2 (1.2) & 0.958     \\
 IFN.y, mean (SD)        &     & 0         & 6.2 (1.4)  & 6.4 (1.1)  & 6.2 (1.6)  & 0.513     \\
 IL.10, mean (SD)        &     & 1         & 2.1 (1.6)  & 2.0 (1.5)  & 2.2 (1.7)  & 0.583     \\
 IL.6, mean (SD)         &     & 0         & 4.7 (1.3)  & 4.3 (1.0)  & 4.9 (1.4)  & 0.048     \\
 IL.8, mean (SD)         &     & 0         & 7.6 (0.9)  & 7.3 (0.7)  & 7.8 (0.9)  & 0.015     \\
 MCP.1, mean (SD)        &     & 0         & 11.4 (0.5) & 11.3 (0.5) & 11.5 (0.5) & 0.127     \\
 MDC, mean (SD)          &     & 0         & 13.6 (0.5) & 13.5 (0.4) & 13.6 (0.5) & 0.281     \\
 MIP.1a, mean (SD)       &     & 3         & 7.4 (1.4)  & 7.6 (1.6)  & 7.3 (1.2)  & 0.36

In [15]:
# checking number of patients not missing data

ndata_NCa_PCa_noMiss = ndata_NCa_PCa.copy()
num_complete_rows = ndata_NCa_PCa_noMiss.dropna().shape[0]
print(num_complete_rows)

39


In [16]:
X = ndata_NCa_PCa.iloc[:,:-1]
y = ndata_NCa_PCa.iloc[:, -1] #.values

# X.head()


In [17]:
def data_split(X,y,rnd_st,tst_sz):
    X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                        test_size = tst_sz, 
                                                        random_state=rnd_st,
                                                        stratify=y)
    
    return X_train, X_test, y_train, y_test

In [18]:
#

X_train, X_test, y_train, y_test = data_split(X,y,rnd_st=1234,tst_sz=0.30)        # working best for now
# X_train, X_test, y_train, y_test = data_split(X,y,rnd_st=1234,tst_sz=0.20)        # working best for now


In [19]:
X_train

,ENA.78,IFN.y,IL.10,IL.6,IL.8,MCP.1,MDC,MIP.1a,TNF.a,C.peptide,G.CSF,IL.22,Insulin,Leptin,MIP.3a,GRO.a,HGF,MMP.2,Adiponectin,CRP,GDF.15,TIMP.1,TGF.B2,TGF.B1,PPAR.y,HIF.1a,Laminin,HbA1c,CA19.9,Glucose,HDL,CCK,LDL,Triglyceride,Albumin,Lumican,ZAG
32,14.156406,7.613148,3.516099,4.657956,7.887444,12.763774,13.796955,7.791547,6.356093,15.262054,6.890430,2.436532,7.780842,17.595141,7.184152,12.583378,9.989478,16.332338,23.008744,22.289698,11.152134,19.208705,6.515822,15.812927,2.109026,NaN,11.839939,9.341799,5.326573,6.257953,9.516256,9.182749,15.142515,4.600508,38.779629,21.158173,21.945726
18,15.012315,6.641013,1.195064,2.914903,7.420325,11.449942,13.482507,6.403229,4.209437,12.449377,6.531318,-1.838386,3.747464,15.155609,5.238005,12.305769,8.819894,14.645313,24.664951,19.091859,9.419016,18.404574,7.358763,16.592913,2.770406,9.427996,10.843401,10.315969,2.796598,5.880514,7.442222,8.866651,14.065079,NaN,38.470049,20.060456,21.980382
43,11.860593,5.548170,1.862631,4.305069,6.248254,10.844231,13.399950,7.334369,5.075557,14.020025,5.029611,NaN,6.395421,18.208958,5.014345,9.854754,8.846574,14.679746,23.802286,19.913486,8.933935,17.806787,7.485568,15.640717,1.893751,8.648857,11.116777,8.729896,8.982138,5.615063,9.250438,7.767125,13.584721,5.841244,38.077162,20.428227,21.805423
54,13.038226,6.696851,4.134884,5.320627,7.717577,11.194252,13.636173,10.781902,5.749328,16.630854,5.303698,2.677300,9.259799,19.890597,6.725526,11.635968,8.930896,15.254802,24.539060,22.301596,10.318429,18.924800,3.600526,16.433337,2.265137,10.341342,10.068231,8.846086,9.223809,6.992859,10.116509,8.409149,14.376717,5.778577,38.267347,20.679643,22.083323
53,12.728181,4.306998,1.820928,4.949380,8.146002,11.267464,13.803969,6.614394,5.468463,13.927260,6.249669,1.523963,4.873964,16.236313,5.592189,11.694441,10.398488,15.464287,24.530897,22.433607,10.517772,18.481488,7.368943,16.083078,0.952334,9.433460,10.935589,8.251719,4.926142,6.043038,10.024763,7.600582,14.625261,5.110447,37.798570,21.221136,21.974735
76,13.253510,6.413213,3.801105,4.459334,7.717063,11.392479,13.478478,7.291582,5.240591,NaN,5.844474,1.220782,NaN,16.884648,6.069991,12.243159,9.399464,16.158590,25.180031,18.991714,9.460934,18.017871,8.330137,15.795065,1.027154,6.997326,11.390926,9.190348,4.508112,7.096652,9.750082,8.050219,13.004181,NaN,37.768698,22.458610,22.356090
42,11.454938,4.269531,0.574165,2.739994,6.545005,10.942439,13.287286,6.035995,4.133665,12.784332,5.952179,-0.395241,4.170913,13.956739,NaN,10.368290,8.414769,14.371736,25.927069,18.616517,9.417047,18.138287,5.286041,15.128574,1.511974,11.188513,10.681577,9.643916,3.935177,6.122652,9.455105,8.446244,13.924549,NaN,38.664720,20.030078,21.986041
65,13.642203,4.518325,0.346054,5.438747,7.503425,11.553704,13.614057,5.880671,4.301901,14.073885,8.477583,1.522862,5.780911,17.366664,5.400644,11.458773,9.740193,14.271023,24.445928,22.575460,10.807129,18.457014,6.003921,16.328062,0.255803,8.784412,9.788250,8.641654,NaN,6.204102,8.726358,8.420874,13.556493,4.644722,37.643296,20.785163,21.747373
2,13.189306,6.382115,3.254247,5.148722,8.136156,11.251296,13.593373,8.306497,5.754229,16.936050,6.784243,1.518603,10.122872,18.310080,7.538368,11.179050,9.322076,15.354492,23.245275,21.586272,9.790895,18.128013,4.594089,16.012930,3.471448,12.146320,10.472401,8.379352,7.538980,7.749313,9.587509,8.105762,15.960987,5.884940,38.009970,21.093460,22.319398
58,12.799916,8.014113,3.506134,7.168899,9.066530,11.239122,13.066945,7.305195,5.941962,13.722590,6.712064,0.740025,5.514380,14.190751,8.119025,11.677251,11.107292,15.300282,23.763651,24.733563,10.781489,19.160561,NaN,15.469029,0.920674,8.181500,11.324175,9.347351,6.887013,6.550885,10.015404,7.881726,14.784897,4.194796,37.868396,20.903713,21.999379


In [20]:
print(X_train.shape)


(56, 37)


In [21]:
# checking number of patients not missing data

X_train_noMiss = X_train.copy()
num_complete_rows = X_train_noMiss.dropna().shape[0]
print(num_complete_rows)

30


In [22]:
# checking number of patients not missing data

X_test_noMiss = X_test.copy()
num_complete_rows = X_test_noMiss.dropna().shape[0]
print(num_complete_rows)

9


In [23]:
# scaler0 = StandardScaler().fit(X_train) # build a scaler for the training data
from sklearn.preprocessing import MinMaxScaler, RobustScaler, MaxAbsScaler

# scaler0 = MinMaxScaler().fit(X_train) # build a scaler for the training data
scaler0 = MaxAbsScaler().fit(X_train) # build a scaler for the training data

In [24]:
# scaled x_train

X_train_sc = scaler0.transform(X_train) # use the scaler to transform the training data


In [25]:
# scaled x_test

X_test_sc = scaler0.transform(X_test) # use the scaler to transform the training data


In [26]:
print(X_train_sc[:,0])

[0.91601238 0.97139528 0.76745823 0.84365877 0.82359684 0.85758911
 0.74120967 0.88274008 0.85343466 0.82823855 1.         0.91163179
 0.90434921 0.86217772 0.77902918 0.84498059 0.98584281 0.84288021
 0.90230096 0.85456926 0.98162433 0.9370071  0.76985619 0.78642332
 0.83513914 0.86906381 0.95161424 0.94647162 0.68605571 0.81840987
 0.89467433 0.82055196 0.83088388 0.87162173 0.81157469 0.84898965
 0.80816429 0.86950558 0.86712861 0.81481859 0.78390489 0.6774645
 0.83676989 0.87026609 0.86065309 0.97530333 0.74553388 0.87162221
 0.94712773 0.82937099 0.72895749 0.94419975 0.84093012 0.88835831
 0.95273077 0.85096247]


In [27]:
X_train_sc.shape

(56, 37)

In [28]:
colnames=X_train.columns.tolist()

# colnames

In [29]:
# X_train

# Conversion with custom column names
X_train_sc_pd = pd.DataFrame(X_train_sc, columns=colnames)


In [30]:
# save to csv file

# 
X_train_sc_pd.to_csv('data/norm_X_train_NCa_PCa.csv', index=False)

In [31]:
# X_test

# Conversion with custom column names
X_test_sc_pd = pd.DataFrame(X_test_sc, columns=colnames)


In [32]:
# save to csv file

# 
X_test_sc_pd.to_csv('data/norm_X_test_NCa_PCa.csv', index=False)

In [33]:
# save to csv file

# 
y_train.to_csv('data/y_train_NCa_PCa.csv', index=False)

In [34]:
# save to csv file

# 
y_test.to_csv('data/y_test_NCa_PCa.csv', index=False)